# Retail Sales & Business Performance Analysis

This notebook uses a **synthetic practice dataset** to demonstrate a beginner data analysis workflow.

## 1. Import libraries and load data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Run from the project root folder
sales = pd.read_csv('data/retail_sales.csv')
sales.head()

## 2. Inspect and clean the data

In [ ]:
print('Rows and columns:', sales.shape)
print('\nMissing values:')
print(sales.isna().sum())

sales['Order_Date'] = pd.to_datetime(sales['Order_Date'], errors='coerce')
for col in ['Quantity', 'Unit_Price', 'Sales', 'Discount', 'Profit']:
    sales[col] = pd.to_numeric(sales[col], errors='coerce')

sales = sales.drop_duplicates()
sales = sales.dropna(subset=['Order_Date', 'Sales', 'Profit'])
print('\nCleaned shape:', sales.shape)

## 3. Calculate key performance indicators

In [ ]:
total_sales = sales['Sales'].sum()
total_profit = sales['Profit'].sum()
profit_margin = total_profit / total_sales if total_sales else 0
units_sold = sales['Quantity'].sum()

kpis = pd.DataFrame({
    'Metric': ['Total Sales', 'Total Profit', 'Profit Margin', 'Units Sold', 'Order Lines'],
    'Value': [round(total_sales, 2), round(total_profit, 2), round(profit_margin * 100, 2), int(units_sold), sales['Order_ID'].nunique()]
})
kpis

## 4. Compare categories

In [ ]:
category_summary = sales.groupby('Category', as_index=False).agg(
    Total_Sales=('Sales', 'sum'), Total_Profit=('Profit', 'sum')
).sort_values('Total_Sales', ascending=False)
category_summary

In [ ]:
category_summary.plot(x='Category', y='Total_Sales', kind='bar', legend=False, title='Total Sales by Category')
plt.ylabel('Sales')
plt.tight_layout()
plt.show()

## 5. Compare regions

In [ ]:
region_summary = sales.groupby('Region', as_index=False).agg(
    Total_Sales=('Sales', 'sum'), Total_Profit=('Profit', 'sum')
).sort_values('Total_Profit', ascending=False)
region_summary

In [ ]:
region_summary.plot(x='Region', y='Total_Profit', kind='bar', legend=False, title='Total Profit by Region')
plt.ylabel('Profit')
plt.tight_layout()
plt.show()

## 6. Monthly sales trend

In [ ]:
sales['Order_Month'] = sales['Order_Date'].dt.to_period('M').astype(str)
monthly = sales.groupby('Order_Month', as_index=False).agg(
    Total_Sales=('Sales', 'sum'), Total_Profit=('Profit', 'sum')
)
monthly.head()

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly['Order_Month'], monthly['Total_Sales'], marker='o')
plt.title('Monthly Sales Trend')
plt.xlabel('Month')
plt.ylabel('Sales')
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()

## 7. Explore discounts and profit

In [ ]:
discount_summary = sales.groupby('Discount', as_index=False).agg(
    Average_Profit=('Profit', 'mean'), Order_Lines=('Order_ID', 'count'), Total_Sales=('Sales', 'sum')
)
discount_summary

## 8. Write business findings

After running the notebook, write 3–5 findings based on the actual outputs. For each finding, include the metric and explain why it matters. Do not assume discounts cause lower profit; this dataset is synthetic and descriptive analysis alone cannot establish causation.